# Avaliação Parcial 1

Problema: Stanford Dogs na Partição do LSIServer7
Arquitetura: EfficientNetB0

1. Uso de Transfer Learning
2. Uso de Taxa de Aprendizado Adaptativa
3. Data Augmentation
4. L2 regularizatiom
5. Early Stopping
6. Gráficos de loss/acurácia no treino e validação
7. Métricas no conjunto de teste
8. Elaboração de um conjunto de slides com os resultados dos experimentos e apresentação da arquitetura
9. Monitoramento do tempo de treinamento
10. Quantidade de parâmetros totais e ajustáveis


#### Grupo

- Adriana Raffaella
- Beatriz Guedes
- Italo Ferreira
- Henrique Furtado

## Bibliotecas

In [3]:
#importando as biblioteca
#OBS: vai falhar na primeira vez, é só rodar de novo
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchsummary import summary
from torchvision import datasets
from torchvision.transforms import v2
import time
import re
import os
import kagglehub
from torch.utils.data import DataLoader
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import random
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import shutil

## Abertura e limpeza do Dataset


In [4]:
path_destino = "./data/dataset_dogs"

if not os.path.exists(path_destino):
  dataset = kagglehub.dataset_download("miljan/stanford-dogs-dataset-traintest")
  shutil.copytree(dataset, "./data/dataset_dogs")
else:
  print("Dataset já existe")

In [5]:
caminho_dataset = "./data/dataset_dogs/cropped"

for raiz, pastas, arquivos in os.walk(caminho_dataset, topdown=False):

  #renomear imagens
  for arquivo in arquivos:
    caminho_antigo = os.path.join(raiz, arquivo)

    novo_nome = re.sub(r"^n\d+[-_ ]*", "", arquivo)
    caminho_novo = os.path.join(raiz, novo_nome)

    if caminho_antigo != caminho_novo:
      os.rename(caminho_antigo, caminho_novo)

  # renomear pastas
  for pasta in pastas:
    caminho_antigo = os.path.join(raiz, pasta)

    novo_nome = re.sub(r"^n\d+[-_ ]*", "", pasta)
    caminho_novo = os.path.join(raiz, novo_nome)

    if caminho_antigo != caminho_novo:
      os.rename(caminho_antigo, caminho_novo)

In [7]:
caminho_treino = "./data/dataset_dogs/cropped/train"
caminho_validacao = "./data/dataset_dogs/cropped/val"

os.makedirs(caminho_validacao, exist_ok=True)

if os.path.exists(caminho_validacao):
  for classe in os.listdir(caminho_treino):
    caminho_classe = os.path.join(caminho_treino, classe)

    if not os.path.isdir(caminho_classe):
      continue

    # cria pasta da classe na validação
    caminho_val_classe = os.path.join(caminho_validacao, classe)
    os.makedirs(caminho_val_classe, exist_ok=True)

    arquivos = os.listdir(caminho_classe)

    n_val = 20
    arquivos_val = random.sample(arquivos, n_val)

    for arquivo in arquivos_val:
      origem = os.path.join(caminho_classe, arquivo)
      destino = os.path.join(caminho_val_classe, arquivo)

      shutil.move(origem, destino)

  print("Divisão concluída.")
else:
  print("Pasta val já existe")

Divisão concluída.


In [8]:
img_size = 224
batch_size = 32

In [9]:
test_transforms = v2.Compose([
  v2.Resize((img_size, img_size)),
  v2.ToTensor(),
  v2.Normalize(
      # Mean e o std com os valores oficiais usados no ImageNet
      mean=[0.485, 0.456, 0.406],
      std=[0.229, 0.224, 0.225]
  )
])

# Data Augmentation
train_transforms = v2.Compose([
    v2.RandomResizedCrop((img_size, img_size)),
    v2.RandomHorizontalFlip(),
    v2.RandomRotation(15),
    v2.ToTensor(),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


#eu não sei se é só essas pastas, vou perguntar da Prof dps
handler_treino = datasets.ImageFolder(
  root="./data/dataset_dogs/cropped/train",
  transform=train_transforms
)

handler_teste = datasets.ImageFolder(
  root="./data/dataset_dogs/cropped/test",
  transform=test_transforms
)

handler_val = datasets.ImageFolder(
  root="./data/dataset_dogs/cropped/val",
  transform=test_transforms
)

#OBS: como q fica o batch size?
DataLoaderTreino = DataLoader(handler_treino, batch_size=batch_size, shuffle=True)

DataLoaderTeste = DataLoader(handler_teste, batch_size=batch_size, shuffle=False)

DataLoaderVal = DataLoader(handler_val, batch_size=batch_size, shuffle=False)

/mnt/c/Users/adraf/Documents/topicos_ativ/.venv/lib/python3.12/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [10]:
print(f"Número de classes: {len(handler_treino.classes)}")
print(f"Imagens de treino: {len(handler_treino)} | Lotes: {len(DataLoaderTreino)}")
print(f"Imagens de teste: {len(handler_teste)} | Lotes: {len(DataLoaderTeste)}")
print(f"Imagens de validação: {len(handler_val)} | Lotes: {len(DataLoaderVal)}")

Número de classes: 120
Imagens de treino: 9600 | Lotes: 300
Imagens de teste: 8580 | Lotes: 269
Imagens de validação: 2400 | Lotes: 75


# Construindo a Arquitetura

## Montagem do Modelo

In [11]:
class Modelo_Adaptado(nn.Module):
  def __init__(self, num_classes=120):
    super().__init__()

    #carregando EfficientNetB0 com os pesos do ImageNet
    self.enbzero = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

    for params in self.enbzero.parameters():
      params.requires_grad = False

    # Aplicando o transfer learning
    in_features = self.enbzero.classifier[1].in_features

    self.enbzero.classifier = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )

  def forward(self, x):
    return self.enbzero(x)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)

modelo = Modelo_Adaptado(num_classes=120).to(device)

Dispositivo: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/adraf/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 23.4MB/s]


In [13]:
summary(model=modelo, input_size=(3, img_size, img_size))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 112, 112]             864
       BatchNorm2d-2         [-1, 32, 112, 112]              64
              SiLU-3         [-1, 32, 112, 112]               0
            Conv2d-4         [-1, 32, 112, 112]             288
       BatchNorm2d-5         [-1, 32, 112, 112]              64
              SiLU-6         [-1, 32, 112, 112]               0
 AdaptiveAvgPool2d-7             [-1, 32, 1, 1]               0
            Conv2d-8              [-1, 8, 1, 1]             264
              SiLU-9              [-1, 8, 1, 1]               0
           Conv2d-10             [-1, 32, 1, 1]             288
          Sigmoid-11             [-1, 32, 1, 1]               0
SqueezeExcitation-12         [-1, 32, 112, 112]               0
           Conv2d-13         [-1, 16, 112, 112]             512
      BatchNorm2d-14         [-1, 16, 1

## Treinamento

In [14]:
criterio = nn.CrossEntropyLoss()

total_params = sum(p.numel() for p in modelo.parameters())
print(f"{total_params:,} parâmetros totais.")
total_trainable_params = sum(
    p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} parâmetros de treino.")

otimizador = optim.Adam(
    modelo.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(otimizador, mode='min', factor=0.1, patience=2)

epocas = 120

historico = []

4,366,324 parâmetros totais.
358,776 parâmetros de treino.


In [15]:
def train(model, train_loader, optimizer, criterion):
  model.train()
  train_loss = 0.0
  train_correct = 0
  total = 0

  for image, labels in train_loader:
    image = image.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    saida = model(image)

    loss = criterion(saida, labels)

    train_loss += loss.item()

    _, previsoes = torch.max(saida, 1)

    train_correct += (previsoes == labels).sum().item()
    total += labels.size(0)

    loss.backward()
    optimizer.step()

  epoch_loss = train_loss / len(train_loader)
  epoch_acc = 100 * (train_correct / total)

  return epoch_loss, epoch_acc

def validate(model, val_loader, criterion):
  model.eval()
  val_correct = 0
  val_loss = 0.0
  total = 0

  with torch.no_grad():
    for image, labels in val_loader:

      image = image.to(device)
      labels = labels.to(device)

      outputs = model(image)

      loss = criterion(outputs, labels)
      val_loss += loss.item()

      _, preds = torch.max(outputs, 1)

      val_correct += (preds == labels).sum().item()
      total += labels.size(0)

  epoch_loss = val_loss / len(val_loader)
  epoch_acc = 100 * (val_correct / total)

  return epoch_loss, epoch_acc

In [16]:
patience = 5
patience_loss = 0
best_loss = float('inf')

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"\n Iniciando treino em: {device_name}")

for epoca in range(epocas):
  start_time = time.time()
  train_epoch_loss, train_epoch_acc = train(modelo, DataLoaderTreino, otimizador, criterio)
  val_epoch_loss, val_epoch_acc = validate(modelo, DataLoaderVal, criterio)

  scheduler.step(val_epoch_loss)

  if val_epoch_loss < best_loss:
      best_loss = val_epoch_loss
      torch.save(modelo.state_dict(), './data/alexnet_dogs_validate.pth')
      patience_loss = 0
  else:
    patience_loss += 1

  if patience_loss >= patience:
    print("Parada antecipada")
    break

  final_time = time.time() - start_time

  print(40*'-')
  print(f"Época {epoca+1}/{epocas}")
  print(f"Train Loss: {train_epoch_loss:.4f} | Train Acc: {train_epoch_acc:.2f}")
  print(f"Val Loss: {val_epoch_loss:.4f} | Val Acc: {val_epoch_acc:.2f}")
  print(f"Time: {final_time:.2f}")
  print(40*'-')

  historico.append([epoca+1, train_epoch_loss, train_epoch_acc, val_epoch_loss, val_epoch_acc])


 Iniciando treino em: NVIDIA GeForce RTX 4050 Laptop GPU
----------------------------------------
Época 1/120
Train Loss: 3.8470 | Train Acc: 14.50
Val Loss: 2.1666 | Val Acc: 48.04
Time: 364.90
----------------------------------------
----------------------------------------
Época 2/120
Train Loss: 2.8559 | Train Acc: 29.30
Val Loss: 1.5914 | Val Acc: 56.83
Time: 131.16
----------------------------------------
----------------------------------------
Época 3/120
Train Loss: 2.6407 | Train Acc: 33.44
Val Loss: 1.4405 | Val Acc: 59.21
Time: 128.09
----------------------------------------
----------------------------------------
Época 4/120
Train Loss: 2.4970 | Train Acc: 36.35
Val Loss: 1.3457 | Val Acc: 62.04
Time: 121.00
----------------------------------------
----------------------------------------
Época 5/120
Train Loss: 2.4138 | Train Acc: 37.82
Val Loss: 1.2489 | Val Acc: 62.71
Time: 122.10
----------------------------------------
----------------------------------------
Época 

In [ ]:
df = pd.DataFrame(historico, columns=["epoca","train_loss","train_acuracia", "validate_loss", "validate_acuracia"])

df.to_csv("historico_treinamento.csv", index=False)